# Phase 8 Stage 1a: Difference-Embedding RSICC — Training & Evaluation

Curriculum stage 1a: frozen **RemoteCLIP ViT-B-32** encodes each fixed
256x256 before/after image independently, a trainable **cross-attention
DifferenceModule** learns the change embedding, and a cheap trainable
**LightweightCaptionDecoder** turns that embedding into a caption. Only the
difference module and decoder are trained; RemoteCLIP stays frozen.

This notebook:
1. Loads LEVIR-CC (train/val/test) using the existing `src/dataset.py` loaders unchanged.
2. Trains with `src/training_phase8.py::train_stage1a`, which **only checkpoints on
   validation-loss improvement** and **auto-resumes** from
   `checkpoints/phase8_stage1a_best.pt` if one already exists (prints the loaded epoch).
3. Evaluates the best checkpoint on **both** the LEVIR-CC test set and the SECOND-CC
   test set using the shared metrics in `src/metrics.py`.

Stage 1b (Qwen2-VL-2B bridge), 1c (joint fine-tune), and stage 2 (per-patch fusion)
are follow-ups once this stage's numbers look reasonable -- not part of this notebook.

In [ ]:
# Cell 1: Environment setup
import os
from pathlib import Path

import torch

from src.utils import get_device, set_seed

print("=" * 70)
print("PHASE 8 STAGE 1a: ENVIRONMENT SETUP")
print("=" * 70)

set_seed(42)
device = get_device()
print(f"[Phase 8] Compute device: {device}")
if device.type == "cuda":
    print(f"[Phase 8] GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[Phase 8] Running on CPU.")

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 2: Config, shared vocabulary, and LEVIR-CC data loaders
# Uses the existing src/dataset.py loaders and transforms unchanged.
from src.config import DataConfig
from src.dataset import build_remoteclip_transforms, get_levircc_loaders
from src.models.phase8 import Phase8Config

data_config = DataConfig()
phase8_config = Phase8Config()

BATCH_SIZE = 8
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4

remoteclip_transforms = build_remoteclip_transforms(
    img_size=phase8_config.img_size,
    mean=phase8_config.remoteclip_mean,
    std=phase8_config.remoteclip_std,
)

train_loader, val_loader, test_loader, vocab = get_levircc_loaders(
    caption_json=data_config.caption_json,
    image_root=data_config.image_root,
    batch_size=BATCH_SIZE,
    device=str(device),
    img_size=phase8_config.img_size,
    num_workers=data_config.num_workers,
    vocab_path=data_config.vocab_path,
    transforms_fn=remoteclip_transforms,
)

print(f"[Phase 8] Vocabulary size: {len(vocab.word2idx)} (loaded/saved at {data_config.vocab_path})")
print(f"[Phase 8] LEVIR-CC train/val/test batches: "
      f"{len(train_loader)}/{len(val_loader)}/{len(test_loader)}")

In [ ]:
# Cell 3: Build the model
from src.models.phase8 import DifferenceRSICCModel

model = DifferenceRSICCModel(vocab=vocab, config=phase8_config)
model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"[Phase 8] Trainable parameters (DifferenceModule + decoder): {trainable_params:,}")
print(f"[Phase 8] Frozen parameters (RemoteCLIP encoder):            {frozen_params:,}")

In [ ]:
# Cell 4: Train stage 1a
# train_stage1a only writes checkpoints/phase8_stage1a_best.pt when val_loss
# improves, and if that file already exists it resumes from it (prints the
# loaded epoch) instead of restarting -- so re-running this cell/notebook is
# always safe to continue a previous run.
from src.training_phase8 import train_stage1a

print("=" * 70)
print("PHASE 8 STAGE 1a: TRAINING")
print("=" * 70)

history = train_stage1a(
    model, train_loader, val_loader, vocab, device,
    epochs=NUM_EPOCHS, lr=LEARNING_RATE,
    checkpoint_dir=CHECKPOINT_DIR, checkpoint_name="phase8_stage1a",
)

In [ ]:
# Cell 5: Load the best checkpoint for evaluation
# (in case the last epoch trained wasn't the best val_loss epoch)
from src.training import load_checkpoint

best_ckpt_path = CHECKPOINT_DIR / "phase8_stage1a_best.pt"
optimizer_for_load = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE
)
model, _, loaded_epoch, loaded_vocab, loaded_val_loss = load_checkpoint(
    model, optimizer_for_load, best_ckpt_path, device
)
model.to(device)
print(f"[Phase 8] Evaluating checkpoint from epoch {loaded_epoch} (val_loss={loaded_val_loss:.4f})")

In [ ]:
# Cell 6: Evaluate on the LEVIR-CC test set
from src.metrics import SentenceEmbeddingScorer, evaluate_full_test_with_per_sample_metrics, save_phase1_results

semantic_scorer = SentenceEmbeddingScorer()

levir_metrics, levir_samples = evaluate_full_test_with_per_sample_metrics(
    model, test_loader, device,
    semantic_model=semantic_scorer,
    max_new_tokens=phase8_config.max_caption_len,
)

print("=" * 70)
print("PHASE 8 STAGE 1a: LEVIR-CC TEST RESULTS")
print("=" * 70)
for k, v in levir_metrics.items():
    print(f"  * {k:<25}: {v:.4f}" if isinstance(v, float) else f"  * {k:<25}: {v}")

save_phase1_results(
    CHECKPOINT_DIR / "phase8_stage1a_levircc_test_results.json",
    levir_metrics, levir_samples,
    metadata={"dataset": "LEVIR-CC", "checkpoint_epoch": loaded_epoch},
)

In [ ]:
# Cell 7: Evaluate on the SECOND-CC test set (zero-shot, no fine-tuning on it)
# Adjust these paths if SECOND-CC-AUG lives elsewhere on your machine/cluster.
from src.dataset import get_secondcc_loaders

SECONDCC_CAPTION_JSON = Path("SECOND-CC-AUG/SECOND-CC-AUG.json")
SECONDCC_IMAGE_ROOT = Path("SECOND-CC-AUG")

if SECONDCC_CAPTION_JSON.is_file():
    _, _, secondcc_test_loader, _ = get_secondcc_loaders(
        caption_json=SECONDCC_CAPTION_JSON,
        image_root=SECONDCC_IMAGE_ROOT,
        vocab=vocab,  # reuse the LEVIR-CC vocabulary so token ids line up
        batch_size=BATCH_SIZE,
        device=str(device),
        img_size=phase8_config.img_size,
        num_workers=data_config.num_workers,
        transforms_fn=remoteclip_transforms,
    )

    secondcc_metrics, secondcc_samples = evaluate_full_test_with_per_sample_metrics(
        model, secondcc_test_loader, device,
        semantic_model=semantic_scorer,
        max_new_tokens=phase8_config.max_caption_len,
    )

    print("=" * 70)
    print("PHASE 8 STAGE 1a: SECOND-CC TEST RESULTS (zero-shot)")
    print("=" * 70)
    for k, v in secondcc_metrics.items():
        print(f"  * {k:<25}: {v:.4f}" if isinstance(v, float) else f"  * {k:<25}: {v}")

    save_phase1_results(
        CHECKPOINT_DIR / "phase8_stage1a_secondcc_test_results.json",
        secondcc_metrics, secondcc_samples,
        metadata={"dataset": "SECOND-CC", "checkpoint_epoch": loaded_epoch},
    )
else:
    secondcc_metrics = None
    print(f"[Phase 8] SECOND-CC-AUG not found at {SECONDCC_CAPTION_JSON.resolve()} -- skipping. "
          "Update SECONDCC_CAPTION_JSON/SECONDCC_IMAGE_ROOT above to point at your copy.")

In [ ]:
# Cell 8: Final combined report
print("=" * 70)
print(f"PHASE 8 STAGE 1a: FINAL METRICS (checkpoint epoch {loaded_epoch}, val_loss={loaded_val_loss:.4f})")
print("=" * 70)
print(f"{'Metric':<25}{'LEVIR-CC test':>18}{'SECOND-CC test':>18}")
for key in levir_metrics:
    levir_v = levir_metrics[key]
    second_v = secondcc_metrics[key] if secondcc_metrics else float('nan')
    if isinstance(levir_v, float):
        print(f"{key:<25}{levir_v:>18.4f}{second_v:>18.4f}")
    else:
        print(f"{key:<25}{levir_v:>18}{'-':>18}")